# Day 1: getting to know the data

**Python for Data Analysis**, Coding for Reproducible Research (CfRR), University of Exeter

## The question

Was summer 2026 actually unusual, or does it just feel that way? Over two sessions we answer it with the Met Office's monthly weather record for Oxford, which starts in 1853. Today we get the data into shape and take a first look. Tomorrow we plot it, fit a trend and bring in a second station.

## By the end of today you will be able to

- read a messy text file into a pandas DataFrame, choosing the separator, header and missing-value markers yourself
- explain how one stray character turns a column of numbers into text, and fix it without throwing information away
- say what a NumPy array is, why vectorised code beats a loop, and how NaN behaves in NumPy and pandas
- decide what to do about missing data, and defend the decision
- summarise data with split-apply-combine (`groupby`)

## How to use this notebook

In the session you type the code yourself, in your own notebook, as the instructor types it on screen. This notebook has every step, with its output and the answer to every exercise. If you fall behind, copy the cell you missed from here into your notebook and carry on. Afterwards, it is your notes.

Exercises are labelled with what they ask of you: **Predict**, **Investigate**, **Modify**, **Spot the bug**, **Describe first**, **Decide together** or **Make**.

The [CfRR Python for Data Analysis pages](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/section_landing_pages/python_for_data_analysis.html) are the full reference. This notebook links to them when you might want every option.

*Data: Met Office historic station data for Oxford. Contains public sector information licensed under the [Open Government Licence v3.0](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).*

> **Instructor: this is your script.** Cells like this one are for you only; they are left out of the learners' solutions notebook. Project a fresh notebook of your own, not this one, and keep this on a second screen.
>
> **Before 10:00, get everyone into a blank notebook:**
>
> 1. Open a terminal in the course folder from the CfRR setup and activate `.venv`.
> 2. Run `pip install plotly`, which the CfRR install command leaves out, then `jupyter lab`.
> 3. **File > New > Notebook**, Python 3 kernel.
> 4. Right-click the tab, **Rename**, call it `day1.ipynb`.
>
> Ask everyone to run `import pandas, plotly` in their first cell and hold up a green sticky note when it works. Put the [solutions link](https://github.com/cafoala/cfrr-python-for-data-analysis/blob/main/notebooks/day1_solutions.ipynb) in the chat or on screen.
>
> **Say:** "You'll type everything yourselves today, as I type it. If you fall behind or something won't run, that link has every cell: copy the one you missed and carry on. Red sticky note if you're stuck and want one of us to come over."

## 1. Welcome, the question and the data

*10:00, 15 minutes*

Before any code: was summer 2026 in Oxford unusual?

**Predict.** Write down your answer, and one number you would want to see before you believed it.

> **Do:** ask for a show of hands: "Who thinks summer 2026 was unusual?" Most will say yes.
>
> **Ask:** "What number would convince you?" Write two or three answers somewhere you can find them on Day 2: the wrap-up comes back to them.
>
> No code in this block. It is talk, and it is full: if it overruns, take the time from the `%timeit` cells in block 5.

### Meet the data

The Met Office publishes [monthly records](https://www.metoffice.gov.uk/research/climate/maps-and-data/historic-station-data) for a set of long-running UK weather stations: one plain text file per station, free to reuse. We will use Oxford, whose record starts in January 1853. The station is at 51.76°N, 1.26°W, 63 metres above sea level.

Each month in the file is summarised by five numbers:

| Column | What it measures | Units |
|---|---|---|
| `tmax` | Mean daily maximum temperature: each day's highest temperature, averaged over the month | °C |
| `tmin` | Mean daily minimum temperature: each night's lowest temperature, averaged over the month | °C |
| `af` | Days of air frost: days when the air temperature fell below 0 °C | days |
| `rain` | Total rainfall for the month | mm |
| `sun` | Total hours of sunshine in the month | hours |

Two things are easy to misread:

- **`tmax` is an average, not a record.** A July `tmax` of 22 °C means the daily highs averaged 22 °C. Some days were much hotter.
- **"Summer" means June, July and August.** That is the meteorologists' definition, and the one we will use.

The record is not perfect, and the file says so:

- **Estimated** values, marked with `*`, fill gaps using the Met Office's gridded dataset (HadUK-Grid) rather than a reading at the station.
- **Missing** values, written `---`, are months with more than two days of data missing.
- **Provisional** values have not yet been through the Met Office's full quality control, and may be revised.
- **Sunshine** was measured for most of the record by a Campbell-Stokes recorder, a glass ball that focuses the sun to burn a trace on a card. Some stations now use an automatic sensor instead.
- Over 170 years, the site and instruments may have changed slightly.

Knowing what each number means, and how it was made, is part of the analysis: it decides which questions the data can answer.

**Predict.** What do you think the average `tmax` is for an Oxford July? We will find out later today.

> **About 8 minutes, talk only.** Put the Met Office station page on screen, or this table.
>
> **Say:** "Before we load anything, we need to know what the numbers mean. Most analysis mistakes happen here, not in the code."
>
> Walk through the five columns, then spend most of the time on the two misreadings:
>
> - **Ask:** "If July's `tmax` is 22 °C, was any day hotter than 22?" (Yes, about half of them: it is an average of the daily highs.)
> - Summer is June to August. Say it now so nobody argues about it later.
>
> Treat the "not perfect" list as a trailer: "Every item on this list turns into code later today." The glass ball usually gets a laugh; it is also why sunshine starts so late.
>
> **Predict, typical July `tmax`:** collect a few guesses. The answer, 22.0 °C, comes out of the seasonal cycle in block 6; point back to the guesses then.

## 2. First look at the raw file

*10:15, 15 minutes*

A notebook runs cells in the order you run them, not the order they appear on the page. Run cells from the top with Shift+Enter. If things get confusing, use **Kernel > Restart Kernel and Run All Cells**.

### The tools

- **pandas** is the standard Python library for tables of data. Its central object is the **DataFrame**: rows and named columns, like a spreadsheet that you drive with code. Every step is written down, so anyone can rerun it and get the same result, which is hard to promise for clicks in a spreadsheet.
- **NumPy** does the fast arithmetic underneath pandas. We look at it properly later this morning.
- **pathlib** builds file locations that work on Windows, macOS and Linux alike.
- **`urlretrieve`** downloads a file. It is part of Python itself, so there is nothing to install.

`import pandas as pd` and `import numpy as np` are conventions: almost all the pandas code you will find elsewhere uses the same short names.

We download the data from the course repository rather than from the Met Office. It is a dated snapshot, so everyone works with exactly the same numbers, and they stay the same when the Met Office updates its own file.

> **Say (one minute, before typing):** "pandas is a spreadsheet you drive with code. That's the whole point of this course: every step written down, so someone else can rerun it." Many learners only know pandas as "the thing you import".
>
> **Type** the next cell a line at a time and say what each one does. Everyone should finish with `True` and a new `data` folder next to their notebook. Show it in the file browser.
>
> **Say:** "The address ends in a date. That's a fixed snapshot, so it can't change under us, unlike the live Met Office file." Long URLs are hard to type: paste this one into the chat.
>
> **If the download fails:**
>
> - `CERTIFICATE_VERIFY_FAILED` on a Mac: Python from python.org needs its certificates installed once. Run **Install Certificates.command** in the Python folder under Applications.
> - No internet, or anything else: download the [fallback zip](https://github.com/cafoala/cfrr-python-for-data-analysis/archive/refs/tags/snapshot-2026-10-29.zip), put the two `.txt` files from its `data/raw` folder into a `data` folder next to the notebook, and delete the `urlretrieve` line.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

BASE = "https://raw.githubusercontent.com/cafoala/cfrr-python-for-data-analysis/snapshot-2026-10-29/"
DATA = Path("data")
DATA.mkdir(exist_ok=True)
# Snapshot of https://www.metoffice.gov.uk/pub/data/weather/uk/climate/stationdata/oxforddata.txt
OXFORD = DATA / "oxforddata.txt"
urlretrieve(BASE + "data/raw/oxforddata.txt", OXFORD)
OXFORD.exists()

True

Look at a data file before you load it. Here are the first ten lines and the last three.

In [2]:
lines = OXFORD.read_text().splitlines()
for line in lines[:10]:
    print(line)
print("...")
for line in lines[-3:]:
    print(line)

Oxford
Location: 450900E 207200N, Lat 51.761 Lon -1.262, 63 metres amsl
Estimated data is marked with a * after the value.
Missing data (more than 2 days missing in month) is marked by  ---.
Sunshine data taken from an automatic Kipp & Zonen sensor marked with a #, otherwise sunshine data taken from a Campbell Stokes recorder.
   yyyy  mm   tmax    tmin      af    rain     sun
              degC    degC    days      mm   hours
   1853   1    8.4     2.7       4    62.8     ---
   1853   2    3.2    -1.8      19    29.3     ---
   1853   3    7.7    -0.6      20    25.9     ---
...
   2026   7   28.2*   14.9*      0*    0.0   353.0*  Provisional
   2026   8   26.5*   14.3*      0*   28.4   267.9*  Provisional
   2026   9   22.2*   12.9*      0*   28.1*  149.1*  Provisional


**Investigate.** Write down three things about this file that might make it awkward to load.

**Answer.** Five lines of free text before the column names; a row of units under the column names; columns separated by spaces rather than commas; `---` for missing values; a `*` after estimated values; the word `Provisional` at the end of recent rows.

> **Do:** give two minutes, then collect answers from the room and type them as a markdown cell in your live notebook. You want all six: free text, units row, spaces not commas, `---`, `*`, `Provisional`. Point at any nobody found; each causes a problem in the next half hour.

**Predict.** What will `pd.read_csv(OXFORD)` give us? How many columns?

In [3]:
naive = pd.read_csv(OXFORD)
naive.shape

(2091, 1)

In [4]:
naive.head()

,,Oxford
Location: 450900E 207200N,Lat 51.761 Lon -1.262,63 metres amsl
Estimated data is marked with a * after the value.,nan,NaN
Missing data (more than 2 days missing in month) is marked by ---.,nan,NaN
Sunshine data taken from an automatic Kipp & Zonen sensor marked with a #,otherwise sunshine data taken from a Campbell Stokes recorder.,NaN
yyyy mm tmax tmin af rain sun,nan,NaN


**Answer.** One column. `read_csv` splits on commas by default and there are none, so each whole line becomes a single value, and the first line (`Oxford`) becomes the column name.

> **Answer:** shape `(2091, 1)`.
>
> **Watch for:** the belief that "CSV" means commas only. **Say:** "`read_csv` reads any delimited text. You just have to tell it how."

## 3. Loading it properly

*10:30, 30 minutes*

"CSV" means comma-separated values, but `read_csv` reads any delimited text once you tell it how. Reading a file is a set of decisions, not a default:

- **`sep`**: what separates the columns
- **`skiprows`**: how many lines to skip before the data
- **`names`**: what to call the columns
- **`na_values`**: how this file writes "missing"

The [CfRR Pandas page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_Pandas.html) covers the other options.

`sep=r"\s+"` means "one or more whitespace characters".

**Investigate.** Run this. Read the error message from the bottom up. What is on the line it complains about?

In [5]:
pd.read_csv(OXFORD, sep=r"\s+")

ParserError: Error tokenizing data. C error: Expected 10 fields in line 4, saw 13


> **Slow down.** The error is `ParserError: Expected 10 fields in line 4, saw 13`.
>
> **Do:** scroll to the bottom of the traceback on screen. **Say:** "Errors are read from the bottom up. The last line says what went wrong; everything above it is where." Many learners read the first line, see a wall of red and give up. This is the moment to fix that habit.
>
> **Ask:** "What's on line 4?" Scroll back up to the printed lines: it is free text.

Line 4 is free text, not data. We will skip all seven lines before the first row of numbers and supply the column names ourselves.

**Modify.** Add `skiprows` and `names` to the call so the first row of the DataFrame is January 1853.

In [6]:
COLUMNS = ["yyyy", "mm", "tmax", "tmin", "af", "rain", "sun"]
pd.read_csv(OXFORD, sep=r"\s+", skiprows=7, names=COLUMNS)

ParserError: Error tokenizing data. C error: Expected 7 fields in line 2084, saw 8


**Investigate.** A new error. Print the line it names (Python counts from 0, so line 2084 is `lines[2083]`). What is different about it?

In [7]:
print(lines[2083])

   2026   1    7.9*    2.1*     11*   34.1    71.3*  Provisional


**Answer.** Recent rows have an eighth field, the word `Provisional`. The data are ragged: some rows are longer than others. That word is information about where the number came from, so we keep it as its own column.

> The error is `ParserError: Expected 7 fields in line 2084, saw 8`. `lines[2083]` is January 2026, the first `Provisional` row.
>
> **Watch for:** people typing `lines[2084]`. **Say:** "pandas counts lines from 1, Python counts from 0."
>
> **Say:** "We could throw that word away. Don't: it tells us these numbers might change. Information about where a number came from is worth keeping." This sets up the whole of block 4.

In [8]:
COLUMNS = ["yyyy", "mm", "tmax", "tmin", "af", "rain", "sun", "status"]
df = pd.read_csv(OXFORD, sep=r"\s+", skiprows=7, names=COLUMNS, na_values="---")
df.head()

,yyyy,mm,tmax,tmin,af,rain,sun,status
0,1853,1,8.4,2.7,4,62.8,NaN,NaN
1,1853,2,3.2,-1.8,19,29.3,NaN,NaN
2,1853,3,7.7,-0.6,20,25.9,NaN,NaN
3,1853,4,12.6,4.5,0,60.1,NaN,NaN
4,1853,5,16.8,6.1,0,59.5,NaN,NaN


In [9]:
df.tail()

,yyyy,mm,tmax,tmin,af,rain,sun,status
2080,2026,5,20.7*,9.6*,0*,12.0,239.1*,Provisional
2081,2026,6,23.8*,14.4*,0*,58.7,231.9*,Provisional
2082,2026,7,28.2*,14.9*,0*,0.0,353.0*,Provisional
2083,2026,8,26.5*,14.3*,0*,28.4,267.9*,Provisional
2084,2026,9,22.2*,12.9*,0*,28.1*,149.1*,Provisional


### Anatomy of a DataFrame

Now the data are in, it is worth knowing the parts:

- the **columns**, each with a name
- the **index**: the row labels down the left (for now just 0, 1, 2 and so on)
- each column on its own is a **Series**: one column of values that keeps the index

**Investigate.** How many rows and columns are there? What does selecting one column give you? Look at the last line of its output, `dtype`: we will need it soon.

In [10]:
df.shape

(2085, 8)

In [11]:
df["tmax"].head(3)

0    8.4
1    3.2
2    7.7
Name: tmax, dtype: str

> `df.shape` is `(2085, 8)`. **Do:** point at the index, the columns and the values in `df.head()` on screen as you name them.
>
> **Ask:** "Look at the last line of the `tmax` output. `dtype: str`. Why would a temperature be text?" (On pandas 2 it says `object`.) Don't answer: block 4 does. Let it bother them.

**Predict.** `na_values="---"` turned the file's own missing marker into proper missing values (`NaN`). Which column do you expect to have the most missing values, and roughly how many?

In [12]:
df.isna().sum()

yyyy         0
mm           0
tmax         0
tmin         1
af           1
rain         0
sun        912
status    2076
dtype: int64

**Answer.** `sun`: sunshine was not recorded at Oxford until 1929. `status` is also mostly missing, but that is expected: only provisional rows have a status. One month, December 1860, is missing `tmin` and `af`.

> **Answers:** `sun` 912, `tmin` 1, `af` 1 (both December 1860), `status` 2076 (only the 9 provisional rows have one).
>
> **Say:** "912 months is 76 years with no sunshine. Hold that thought until after the break."

## 4. The asterisk problem

*11:00, 30 minutes*

**Predict.** What will `df["tmax"].mean()` return?

In [13]:
df["tmax"].mean()

TypeError: Cannot perform reduction 'mean' with string dtype

> A `TypeError`, worded differently by version:
>
> - pandas 3: `Cannot perform reduction 'mean' with string dtype`
> - pandas 2 (Python 3.10): `Could not convert string '8.43.27.7...' to numeric`. **Say:** "pandas glued every value into one long string and tried to read it as a number." That shows the column really is text.

**Investigate.** Check the data type of each column. Why is `tmax` not a number?

In [14]:
df.dtypes

yyyy      int64
mm        int64
tmax        str
tmin        str
af          str
rain        str
sun         str
status      str
dtype: object

`tmax` is stored as text (`str` in pandas 3, `object` in earlier versions). One value pandas cannot read as a number is enough to make the whole column text. Here are the culprits:

In [15]:
df[df["tmax"].str.endswith("*")].tail()

,yyyy,mm,tmax,tmin,af,rain,sun,status
2080,2026,5,20.7*,9.6*,0*,12.0,239.1*,Provisional
2081,2026,6,23.8*,14.4*,0*,58.7,231.9*,Provisional
2082,2026,7,28.2*,14.9*,0*,0.0,353.0*,Provisional
2083,2026,8,26.5*,14.3*,0*,28.4,267.9*,Provisional
2084,2026,9,22.2*,12.9*,0*,28.1*,149.1*,Provisional


The `*` marks an estimated value. That matters: July 2026, the month we care about, is one of them.

**Investigate.** An AI assistant suggests `pd.to_numeric(df["tmax"], errors="coerce")` to fix the column. Predict what it does to `28.2*`, then run it and count the missing values.

In [16]:
suggested = pd.to_numeric(df["tmax"], errors="coerce")
suggested.isna().sum()

np.int64(26)

**Answer.** `errors="coerce"` turns anything it cannot read into `NaN`, without telling you. Every estimated value disappears, including July 2026. The code runs and the answer is wrong. That is the worst kind of bug.

> **Slow down. This is the most important idea of the day.** 26 values become `NaN`, including July 2026.
>
> **Ask:** "Hands up if you've used `errors="coerce"`." Several will have; it is a common Stack Overflow and Copilot answer.
>
> **Say:** "No error, no warning, and the month we care about is gone. Code that runs and gives the wrong answer is the worst kind of bug. We want code that fails loudly."

A better approach keeps the information: record which values were estimated in a column of its own, then strip the `*` and convert. `.astype(float)` fails loudly if it meets anything else it does not expect, which is what we want.

In [17]:
# Keep the flag, then strip the marker and convert
estimated = df["tmax"].str.endswith("*")
tmax_clean = df["tmax"].str.rstrip("*").astype(float)
estimated.sum(), tmax_clean.mean()

(np.int64(26), np.float64(14.029112709832134))

All five measured columns need the same treatment, and the results should go back into `df`. A loop does it once for all of them.

`astype(str)` first makes sure every column is text before we look for a `*`; at some stations a column has no estimates and pandas reads it as numbers already.

**Modify.** Turn the cell above into a loop over `MEASURES` that replaces each column in `df` with its cleaned version and adds an `_estimated` column alongside it.

> The single-column cell gives 26 estimated values and a mean `tmax` of 14.03 °C.
>
> **Why `astype(str)` in the loop:** at some stations a column with no estimates is read as numbers, and `.str` fails on numbers. Lerwick, tomorrow, is one of them. Say so, but don't demonstrate it.
>
> The loop is the longest thing typed today. Let learners try it for five minutes, then type it yourself slowly. Remind people it is in the solutions notebook.
>
> **Watch for:** "estimated means wrong, so drop those rows". **Say:** "Keep the flag, decide later. Once a row is gone you can't change your mind."

In [18]:
MEASURES = ["tmax", "tmin", "af", "rain", "sun"]
for col in MEASURES:
    text = df[col].astype(str)
    df[col + "_estimated"] = text.str.endswith("*")
    df[col] = text.str.rstrip("*").astype(float)
df.dtypes

yyyy                int64
mm                  int64
tmax              float64
tmin              float64
af                float64
rain              float64
sun               float64
status                str
tmax_estimated       bool
tmin_estimated       bool
af_estimated         bool
rain_estimated       bool
sun_estimated        bool
dtype: object

**Make.** Replace the `status` column with a `True`/`False` column called `provisional`, then drop `status`.

In [19]:
df["provisional"] = df["status"] == "Provisional"
df = df.drop(columns="status")
df[["yyyy", "mm", "tmax", "tmax_estimated", "provisional"]].tail(3)

,yyyy,mm,tmax,tmax_estimated,provisional
2082,2026,7,28.2,True,True
2083,2026,8,26.5,True,True
2084,2026,9,22.2,True,True


We have kept every flag as its own column rather than deleting the estimated and provisional rows. Throwing them away would have thrown away July 2026.

## Break

*11:30, 15 minutes*

> **Before the break, say:** "If your notebook is in a mess, that's fine. After the break, copy anything you're missing from the solutions notebook so you have `df` with the `_estimated` columns." Check for red sticky notes while people are out.

## 5. What's underneath: NumPy

*11:45, 35 minutes*

Every pandas column is a NumPy array with a name and an index attached. NumPy is what makes the arithmetic fast. The [CfRR NumPy page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_NumPy.html) has the full tour.

In [20]:
tmax = df["tmax"].to_numpy()
tmin = df["tmin"].to_numpy()
type(tmax), tmax.dtype, tmax.shape

(numpy.ndarray, dtype('float64'), (2085,))

> `numpy.ndarray`, `float64`, `(2085,)`. **Say:** "Underneath, a column is just this: a block of numbers, all the same type. No names, no index. That's why it's fast."

The mean temperature for a month is the average of its mean maximum and mean minimum. Here it is the way the Introduction to Python course would write it, with a loop:

In [21]:
tmean_loop = []
for i in range(len(tmax)):
    tmean_loop.append((tmax[i] + tmin[i]) / 2)
tmean_loop[:3]

[np.float64(5.550000000000001),
 np.float64(0.7000000000000001),
 np.float64(3.5500000000000003)]

**Modify.** Write the same calculation without a loop. NumPy applies arithmetic to every element at once: this is called vectorisation.

In [22]:
tmean = (tmax + tmin) / 2
tmean[:3]

array([5.55, 0.7 , 3.55])

**Investigate.** `%timeit` runs a line many times and reports how long it takes. How much faster is the vectorised version?

In [23]:
%timeit [(tmax[i] + tmin[i]) / 2 for i in range(len(tmax))]

328 μs ± 3.93 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [24]:
%timeit (tmax + tmin) / 2

1.71 μs ± 19.4 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


> On the instructor laptop: about 318 µs for the loop and 1.7 µs vectorised, roughly 190 times faster. Learners' numbers will differ; the ratio is the point.
>
> **If block 1 overran, skip these two cells** and just say the ratio.

### How NaN behaves

December 1860 has no `tmin`. **Predict** what each of these three lines returns before you run them.

In [25]:
np.mean(tmin)

np.float64(nan)

In [26]:
np.nanmean(tmin)

np.float64(6.280278310940498)

In [27]:
df["tmin"].mean()

np.float64(6.280278310940498)

**Answer.** `np.mean` returns `nan`: any arithmetic with `NaN` gives `NaN`, so one missing month spoils the whole result. `np.nanmean` skips missing values. pandas skips them by default. Neither choice is wrong, but you need to know which one you are getting.

> `nan`, then 6.28, then 6.28. Run them one at a time and take a guess from the room before each.
>
> **Watch for:** "NaN is zero". **Say:** "NaN means unknown. Unknown plus anything is unknown."

Add the monthly mean temperature to the DataFrame. pandas lines the values up by row for us.

In [28]:
df["tmean"] = (df["tmax"] + df["tmin"]) / 2

### Boolean masks

A comparison on a column gives a column of `True`/`False` values, which can pick out rows.

In [29]:
frost_free = df["af"] == 0
frost_free.sum()

np.int64(1009)

**Modify.** Use the mask to find which months of the year are most often frost-free. (Hint: select the `mm` column of the frost-free rows, then `.value_counts()`.)

In [30]:
df.loc[frost_free, "mm"].value_counts()

mm
6     174
7     174
8     174
9     173
5     132
10     93
4      43
11     18
12      8
2       8
3       8
1       4
Name: count, dtype: int64

> 1009 frost-free months. Every June, July and August is frost-free (174 each), and every September bar one (173).
>
> **Do:** show `frost_free.head()` first so people see the column of `True`/`False` before using it to pick rows.

### Decide together: the missing sunshine

**Predict.** `df.dropna()` removes every row with a missing value. How many of the rows will survive?

In [31]:
len(df), len(df.dropna())

(2085, 1173)

Sunshine is missing for every month from 1853 to 1928. Dropping those rows also throws away 76 years of perfectly good temperature records.

**Decide together.** What should we do about the missing sunshine? Some options:

1. Drop every row with a missing value
2. Drop the `sun` column
3. Fill the gaps with the average sunshine
4. Keep everything, leave the gaps as `NaN`, and only use sunshine from 1929

What does each one cost? Does the answer change for the single missing month of `tmin`?

> **Slow down. Let the room argue; this is the interactive moment of the day.** `df.dropna()` keeps 1173 of 2085 rows.
>
> **Do:** copy the options cell above into your live notebook so it is on screen. Take each option in turn and ask "What does this one cost?" before revealing anything:
>
> 1. dropping rows loses 76 years of temperature
> 2. dropping the column loses the sunshine we do have
> 3. filling invents data: run the next cell to show the standard deviation of `sun` falling from 63.9 to 47.9 hours
> 4. **recommended:** keep everything, leave the gaps as `NaN`, use sunshine from 1929
>
> For the single missing `tmin`, any sensible answer is fine as long as it is stated. **Say:** "There's no single right answer. There is a wrong way to do it: deciding without saying so."

**Recommended.** Option 4. pandas already skips `NaN` in calculations, so the gaps do no harm as long as we know they are there. Option 1 loses most of the temperature record; option 2 loses the sunshine we do have. Option 3 is the most dangerous: it invents 76 years of perfectly average sunshine, as the next cell shows. For the single missing `tmin`, leaving it as `NaN` is fine too; filling one month in a long record would also be defensible, as long as you say so.

In [32]:
filled = df["sun"].fillna(df["sun"].mean())
df["sun"].std(), filled.std()

(np.float64(63.86273077602531), np.float64(47.89194042558409))

Filling makes the record look far less variable than it was. Any analysis of sunshine over time would be quietly wrong.

## 6. Summarising: split-apply-combine

*12:20, 30 minutes*

First, give each row a proper date. A datetime index lets pandas understand months and years.

In [33]:
df.index = pd.to_datetime({"year": df["yyyy"], "month": df["mm"], "day": 1})
df[["yyyy", "mm", "tmax", "tmean"]].head(3)

,yyyy,mm,tmax,tmean
1853-01-01,1853,1,8.4,5.55
1853-02-01,1853,2,3.2,0.70
1853-03-01,1853,3,7.7,3.55


**Describe first.** Before writing any code, write in plain English how you would work out the average summer (June to August) maximum temperature for each year.

**Answer.** Keep only the June, July and August rows. Split them into groups by year. Take the mean of `tmax` in each group. Combine the results into one table.

> **Do:** insist on the English before the code. Take a plan from the room and type it as a markdown cell in your live notebook, one step per line: keep June to August, split by year, mean of `tmax`, combine. Then write the code under it, one line per step.

That plan is split-apply-combine, and `groupby` does it in one line.

In [34]:
summer = df[df["mm"].isin([6, 7, 8])]
summer_tmax = summer.groupby("yyyy")["tmax"].mean()
summer_tmax.tail()

yyyy
2022    24.666667
2023    22.266667
2024    21.966667
2025    24.500000
2026    26.166667
Name: tmax, dtype: float64

**Modify.** Change the grouping from year to month, using all rows rather than just summer. What does the result now mean?

In [35]:
df.groupby("mm")["tmax"].mean()

mm
1      6.827011
2      7.586782
3     10.123563
4     13.330460
5     16.853448
6     20.054598
7     21.959770
8     21.400575
9     18.658621
10    14.270520
11     9.778613
12     7.444509
Name: tmax, dtype: float64

**Answer.** The average maximum temperature for each month of the year over the whole record: the seasonal cycle. Compare July with your guess from the start of the session.

> The seasonal cycle runs from 6.8 °C in January to 22.0 °C in July. **Do:** go back to the July guesses from 10:00. Who was closest?

**Make.** Rank every July in the record by `tmax`, hottest first. Show the top five, with their `tmax_estimated` and `provisional` flags.

In [36]:
julys = df[df["mm"] == 7].sort_values("tmax", ascending=False)
julys[["tmax", "tmax_estimated", "provisional"]].head()

,tmax,tmax_estimated,provisional
2026-07-01,28.2,True,True
2018-07-01,27.4,False,False
2006-07-01,27.1,False,False
1983-07-01,26.8,False,False
1911-07-01,26.1,False,False


> Top five Julys: 2026 28.2 °C (estimated and provisional), 2018 27.4, 2006 27.1, 1983 26.8, 1911 26.1.
>
> **Say:** "Look at the flags on the top row. The hottest July on record is a number the Met Office estimated and hasn't finished checking. Good thing we kept the flags."

## 7. Cliffhanger

*12:50, 10 minutes*

Where does summer 2026 come in the ranking of whole summers?

In [37]:
summer_tmax.sort_values(ascending=False).head()

yyyy
2026    26.166667
1976    24.900000
2022    24.666667
2018    24.533333
2025    24.500000
Name: tmax, dtype: float64

So July 2026 is the hottest July on record, and summer 2026 the hottest summer. Is that enough to say it was unusual?

- The 2026 temperatures are estimated and provisional.
- In any long record, some year has to be the hottest. Is 2026 far ahead, or just a bit ahead?
- Summers have been getting warmer. Is 2026 unusual for its time, or just the latest step on a trend?

Tomorrow we plot it, fit a trend and find out.

### Today's ideas

- Reading a file is a set of decisions: `sep`, `skiprows`, `names`, `na_values`.
- One stray character turns a numeric column into text. Fix it and keep the information as a flag.
- `errors="coerce"` hides problems. Prefer code that fails loudly.
- A column is a NumPy array. Vectorised arithmetic beats loops.
- `NaN` spreads in NumPy and is skipped in pandas.
- Dropping or filling missing data is a decision with consequences.
- `groupby` is split-apply-combine.

> Summer ranking: 2026 26.2 °C, 1976 24.9, 2022 24.7, 2018 24.5, 2025 24.5.
>
> **Do not answer the question.** Copy the three doubts onto screen and say: "That's tomorrow's agenda."
>
> **Say:** "Save your notebook. Tomorrow you start with a fresh one, and a single line loads today's cleaned data, so it doesn't matter if today's notebook is messy."